# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the `mlcroissant` library to load, explore, and process the FAIR² human extracellular vesicle proteomics dataset described by a Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(url)

# Show the high-level dataset metadata (name, description)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and fields. Entities (record sets, fields, and columns) are referenced by their `@id`, per FAIR-Croissant best practices.

In [ ]:
# Function to print record sets and their fields by @id
all_record_sets = dataset.metadata.record_sets

print(f"Found {len(all_record_sets)} record sets:\n")
for rec_set in all_record_sets:
    print(f"RecordSet @id: {rec_set.id}")
    print(f"  Name: {getattr(rec_set, 'name', '(none)')}")
    print("  Fields:")
    for f in rec_set.fields:
        print(f"    - Field @id: {f.id}, Name: {getattr(f, 'name', '(none)')}, Data type: {getattr(f, 'data_type', '(none)')}")
    print()

## 3. Data Extraction
Let's load data from a record set. We'll use the `@id` for the main protein record set and field(s) identified above.

> **Note**: Adjust the `record_set_ids` list if your analysis targets other record sets.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Extract and load all records into DataFrames, using the @id for referencing
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id '{record_set_id}'\nColumns: {list(df.columns)}\n")

# Choose first record set as main for further steps
main_record_set_id = record_set_ids[0]
print(f"Previewing first 5 records of record set: {main_record_set_id}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic statistical and filtering operations using `@id`-referenced fields.

We'll:
- Select a numeric field (`@id`) for filtering & normalization.
- Filter records with high values.
- Normalize and group by another field.

In [ ]:
# Identify a numeric field @id for analysis
main_field_ids = [f.id for f in dataset.metadata["record_sets"][0].fields]
print(f"Available field @ids for main record set: {main_field_ids}")
# For this dataset, mass spectrometry analysis typically includes fields like "Molecular Weight" (MW) or "Abundance"
# Let's try to select 'cr:MW' as numeric field -- double-check your @id via previous overview step

# Replace with an actual field @id present in your dataset
numeric_field_id = None
for f in dataset.metadata["record_sets"][0].fields:
    if "mw" in f.id.lower() or "abundance" in f.id.lower() or (getattr(f, 'data_type', '').lower() in ("float", "number", "integer")):
        numeric_field_id = f.id
        break
if numeric_field_id is None:
    # Fallback: use first available
    numeric_field_id = main_field_ids[0]

print(f"Selected numeric field @id: {numeric_field_id}")
threshold = 10
df = dataframes[main_record_set_id].copy()

# Cast column to numeric, errors='coerce' handles strings
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
print(filtered_df.head(3))

# Normalization
colnorm = f"{numeric_field_id}_normalized"
filtered_df[colnorm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, colnorm]].head(3))

# Try to group by a likely categorical field
cat_field_id = None
for f in dataset.metadata["record_sets"][0].fields:
    if f.id != numeric_field_id and getattr(f, 'data_type', '').lower() in ("string", "text"):
        cat_field_id = f.id
        break
if cat_field_id and cat_field_id in filtered_df:
    grouped_df = filtered_df.groupby(cat_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped average {numeric_field_id} by {cat_field_id}:")
    print(grouped_df.head(5))

## 5. Visualization
We can plot the distribution of a numeric field using matplotlib (histogram), and show any clear group-level differences.

In [ ]:
import matplotlib.pyplot as plt

# Histogram for the selected numeric field
plt.figure(figsize=(8, 4))
filtered_df[numeric_field_id].hist(bins=25)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Optionally boxplot of normalized values by group
if cat_field_id and cat_field_id in filtered_df:
    plt.figure(figsize=(10, 5))
    filtered_df.boxplot(column=colnorm, by=cat_field_id, rot=45)
    plt.title(f"Normalized {numeric_field_id} by {cat_field_id}")
    plt.suptitle("")
    plt.ylabel(colnorm)
    plt.xlabel(cat_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load a complex Croissant-FAIR dataset package, inspect its schema (with record set and field `@id`s), load data, and conduct simple exploratory analysis directly referencing the dataset entities. All access was via `@id` for clarity and reproducibility.

Further work can include more in-depth biological analysis, leveraging record-level links for annotation and integrating with public protein/proteomics resources. The `mlcroissant` library facilitates robust FAIR data workflows for bioinformatics and beyond.